# AI Arena — Track 1 Starter Notebook

**Hallucination Hunter** — given a claim, predict whether it is `supported`, `refuted`, or `not_enough_info` based on the evidence the test harness provides.

Submit your predictions and the platform will return your macro-F1 score and a per-class breakdown. See [`docs/judge_rubrics.md`](judge_rubrics.md) for the full scoring rules.

**Daily limit:** 5 submissions per student. Use them wisely.

---

## What you'll do in this notebook

1. Read the public sample claims (we'll release the hidden test set on launch day).
2. Plug in your prediction strategy — anything from a simple regex to a full LLM call.
3. Format predictions into the AI Arena submission envelope.
4. POST it to the platform and read your score back.

## 0. Setup

In [ ]:
# Install the only thing we need: a request library.
%pip install -q httpx pydantic

import httpx, json, datetime, uuid

# --- Configure these ---
ARENA_URL = 'https://d4-arena-api-XXXXX.run.app'  # ask your instructor for the real URL
STUDENT_ID = 'asukul'                              # your Canvas/NetID
MODEL_USED = 'manual'                              # e.g. 'claude-haiku-4-5', 'manual', 'gemini-flash'
PROMPT_VERSION = 'v1'
STRATEGY_NOTES = 'First attempt: keyword matching baseline.'

## 1. The sample claims (replace with the real test set on launch day)

The real test set will arrive as a JSON file or via a download link from the instructor. For now, here are five sample claims so you can exercise the workflow.

In [ ]:
claims = [
    {'claim_id': 'C001', 'text': 'Iowa State University was founded in 1858.'},
    {'claim_id': 'C002', 'text': 'The Cy-Hawk trophy is awarded to the loser of the ISU vs Iowa football game.'},
    {'claim_id': 'C003', 'text': 'The Memorial Union has more square feet than the Parks Library.'},
    {'claim_id': 'C004', 'text': 'Iowa State established the first U.S. graduate program in artificial intelligence.'},
    {'claim_id': 'C005', 'text': 'The ISU mascot Cy was introduced in 1954.'},
]

## 2. Your prediction strategy

Replace the body of `predict()` with your approach. The function must return one of `supported`, `refuted`, or `not_enough_info`.

In [ ]:
def predict(claim_text: str) -> str:
    # --- BASELINE: predict 'not_enough_info' for everything. ---
    # Replace this with your own logic — keyword rules, LLM call, retrieval, etc.
    return 'not_enough_info'

# Sanity check
for c in claims[:2]:
    print(c['claim_id'], '→', predict(c['text']))

## 3. Build the AI Arena submission envelope

Every submission to the Arena follows the same JSON shape — see [`AI_Arena_Specification.docx` §4](../AI_Arena_Specification.docx). Track 1's `track_payload` is just a `predictions` list.

In [ ]:
predictions = [
    {'claim_id': c['claim_id'], 'label': predict(c['text'])}
    for c in claims
]

envelope = {
    'submission_id': f'sub_{STUDENT_ID}_{uuid.uuid4().hex[:8]}',
    'student_id': STUDENT_ID,
    'track_id': 'hallucination_hunter',
    'submission_timestamp': datetime.datetime.now(datetime.UTC).isoformat(),
    'model_used': MODEL_USED,
    'prompt_version': PROMPT_VERSION,
    'self_reported_strategy': STRATEGY_NOTES,
    'track_payload': {'predictions': predictions},
}

print(json.dumps(envelope, indent=2))

## 4. Submit and read your score

POST to `/submit`. The response tells you: did it queue? How many of today's 5 submissions have you used?

Then poll `/leaderboard/hallucination_hunter` to see where you landed.

In [ ]:
with httpx.Client(timeout=30.0) as client:
    response = client.post(f'{ARENA_URL}/submit', json=envelope)

if response.status_code == 200:
    body = response.json()
    print('Submitted!')
    print(f"  submissions_today = {body['submissions_today']}/{body['daily_limit']}")
    print(f"  task_id           = {body['task_id']}")
elif response.status_code == 422:
    print('Your submission did not validate. Fix these problems:')
    for p in response.json()['problems']:
        print(f"  - {p['field']}: {p['problem']}")
elif response.status_code == 429:
    detail = response.json()['detail']
    print(f"You've hit the daily limit ({detail['current']}/{detail['limit']}). Resets at {detail['reset_at']}.")
else:
    print('Unexpected status:', response.status_code, response.text)

In [ ]:
# Read your current spot on the leaderboard. Score is your best across
# all submissions today — not your latest.
leaderboard = httpx.get(f'{ARENA_URL}/leaderboard/hallucination_hunter').json()
for rank, entry in enumerate(leaderboard['entries'][:10], start=1):
    marker = ' <-- you' if entry['student_id'] == STUDENT_ID else ''
    print(f"{rank:>3}. {entry['student_id']:<20} {entry['final_score']:.4f}{marker}")

## 5. What to try next

- **Plug in an LLM**: replace `predict()` with a Claude/Gemini/OpenAI call.
- **Retrieve evidence first**: pull the relevant sentence from a source corpus before asking the LLM. Even a simple BM25 retriever often beats raw LLM knowledge.
- **Read your per-class breakdown**: from `judge_metadata.per_class`, find the class with the lowest F1 and focus there.
- **Watch your costs**: `model_used` and `self_reported_strategy` are recorded in your submission — they help us all learn what works.

Good luck!